# 01 — Data Exploration
**Apple Disease & Rot Detection — Track A (Yacine)**

This notebook explores the two datasets that feed our model:
- **NVIDIA Fruits dataset** → `healthy_fruit`, `rotten_fruit` classes
- **Plant Pathology 2021** → leaf disease classes (`apple_scab`, `apple_black_rot`, etc.)

Run every cell top-to-bottom. Set the path variables in Section 1 before running.

## 1. Configuration — Set Your Paths Here

In [ ]:
from pathlib import Path

# ── EDIT THESE TWO PATHS ──────────────────────────────────────────────────────

# Path to the NVIDIA fruits dataset (the folder that contains train/ and valid/)
NVIDIA_FRUITS_PATH = Path(r"E:\NVIDIA TRAINING ICME\data\fruits")

# Path to your Plant Pathology 2021 download
# After downloading from Kaggle, point this at the folder with train/ images
PLANT_PATH = Path(r"C:\path\to\plant-pathology-2021-fgvc8")  # <-- update this

# ─────────────────────────────────────────────────────────────────────────────

print(f"NVIDIA fruits path exists: {NVIDIA_FRUITS_PATH.exists()}")
print(f"Plant Pathology path exists: {PLANT_PATH.exists()}")

In [ ]:
import os
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print("Imports OK")

---
## 2. Our Class Taxonomy
These are the exact class names defined in `docs/api-contract.md`. Both datasets will be mapped to these names.

In [ ]:
# Exact class names from docs/api-contract.md — do NOT change these
LEAF_DISEASES = [
    "apple_scab",
    "apple_black_rot",
    "cedar_apple_rust",
    "powdery_mildew",
    "frog_eye_leaf_spot",
    "healthy_leaf",
]

FRUIT_CONDITIONS = [
    "rotten_fruit",
    "bruised_fruit",
    "healthy_fruit",
]

ALL_CLASSES = LEAF_DISEASES + FRUIT_CONDITIONS
print(f"Total classes: {len(ALL_CLASSES)}")
for i, c in enumerate(ALL_CLASSES):
    print(f"  [{i}] {c}")

---
## 3. NVIDIA Fruits Dataset
Source: Kaggle 'Fruits Fresh and Rotten for Classification'  
Obtained via: NVIDIA DLI training (07_assessment notebook)  
Relevant classes: **freshapples → healthy_fruit**, **rottenapples → rotten_fruit**

In [ ]:
# Map NVIDIA class names → our taxonomy (only apple classes are kept)
NVIDIA_CLASS_MAP = {
    "freshapples":   "healthy_fruit",
    "rottenapples":  "rotten_fruit",
    # Skipped (not apple-related):
    # freshbanana, freshoranges, rottenbanana, rottenoranges
}

nvidia_stats = []

for split in ["train", "valid"]:
    split_path = NVIDIA_FRUITS_PATH / split
    if not split_path.exists():
        print(f"WARNING: {split_path} not found — check NVIDIA_FRUITS_PATH")
        continue
    for cls_folder in sorted(split_path.iterdir()):
        if not cls_folder.is_dir():
            continue
        images = list(cls_folder.glob("*.png")) + list(cls_folder.glob("*.jpg"))
        our_class = NVIDIA_CLASS_MAP.get(cls_folder.name, "(skipped)")
        nvidia_stats.append({
            "split": split,
            "original_class": cls_folder.name,
            "our_class": our_class,
            "count": len(images)
        })

nvidia_df = pd.DataFrame(nvidia_stats)
print(nvidia_df.to_string(index=False))

In [ ]:
# Plot: NVIDIA class distribution (apple classes only)
apple_nvidia = nvidia_df[nvidia_df["our_class"] != "(skipped)"]

if not apple_nvidia.empty:
    pivot = apple_nvidia.pivot(index="our_class", columns="split", values="count").fillna(0)
    pivot.plot(kind="bar", color=["#4C9BE8", "#F4A261"], edgecolor="white")
    plt.title("NVIDIA Fruits Dataset — Apple Classes", fontsize=14, fontweight='bold')
    plt.xlabel("Class")
    plt.ylabel("Image Count")
    plt.xticks(rotation=0)
    plt.legend(title="Split")
    plt.tight_layout()
    plt.show()
    print(f"\nTotal apple images from NVIDIA: {apple_nvidia['count'].sum()}")
else:
    print("No data — check that NVIDIA_FRUITS_PATH is correct")

In [ ]:
# Show sample images — 4 per apple class from train set
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("NVIDIA Fruits Dataset — Sample Images", fontsize=14, fontweight='bold')

row = 0
for orig_cls, our_cls in NVIDIA_CLASS_MAP.items():
    cls_path = NVIDIA_FRUITS_PATH / "train" / orig_cls
    if not cls_path.exists():
        continue
    images = list(cls_path.glob("*.png")) + list(cls_path.glob("*.jpg"))
    samples = random.sample(images, min(4, len(images)))
    for col, img_path in enumerate(samples):
        ax = axes[row][col]
        img = Image.open(img_path)
        ax.imshow(img)
        ax.set_title(f"{our_cls}\n{img.size[0]}x{img.size[1]}", fontsize=9)
        ax.axis('off')
    row += 1

plt.tight_layout()
plt.show()

In [ ]:
# Image size distribution for NVIDIA apple images
widths, heights = [], []
for orig_cls in NVIDIA_CLASS_MAP:
    for split in ["train", "valid"]:
        cls_path = NVIDIA_FRUITS_PATH / split / orig_cls
        if not cls_path.exists():
            continue
        for img_path in cls_path.glob("*.png"):
            try:
                w, h = Image.open(img_path).size
                widths.append(w)
                heights.append(h)
            except Exception:
                pass

if widths:
    print(f"Width  — min: {min(widths)}  max: {max(widths)}  mean: {np.mean(widths):.0f}")
    print(f"Height — min: {min(heights)}  max: {max(heights)}  mean: {np.mean(heights):.0f}")
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(widths, bins=20, color='#4C9BE8', edgecolor='white')
    axes[0].set_title('Image Widths')
    axes[0].set_xlabel('Pixels')
    axes[1].hist(heights, bins=20, color='#F4A261', edgecolor='white')
    axes[1].set_title('Image Heights')
    axes[1].set_xlabel('Pixels')
    plt.tight_layout()
    plt.show()

---
## 4. Plant Pathology 2021
Source: Kaggle — Plant Pathology 2021 - FGVC8  
Covers all **leaf disease** classes in our taxonomy.

The dataset has a `train_images/` folder and `train.csv` with multi-label annotations.  
Single-label images (one disease per photo) are the cleanest for our classifier.

In [ ]:
# Plant Pathology 2021 class mapping
# Dataset labels → our taxonomy
PP2021_CLASS_MAP = {
    "scab":              "apple_scab",
    "rot":               "apple_black_rot",
    "rust":              "cedar_apple_rust",
    "powdery_mildew":    "powdery_mildew",
    "frog_eye_leaf_spot": "frog_eye_leaf_spot",
    "healthy":           "healthy_leaf",
    # 'complex' = multi-label images — we skip these for now
}

# Load the CSV
csv_path = PLANT_PATH / "train.csv"
if not csv_path.exists():
    print(f"WARNING: {csv_path} not found. Update PLANT_PATH above.")
else:
    pp_df = pd.read_csv(csv_path)
    print(f"Loaded {len(pp_df)} rows from train.csv")
    print(pp_df.head())
    print("\nColumns:", pp_df.columns.tolist())

In [ ]:
# Parse multi-label column (Plant Pathology 2021 uses space-separated labels)
try:
    # The 'labels' column contains space-separated disease names
    label_col = "labels"  # adjust if column name differs
    pp_df["label_list"] = pp_df[label_col].str.split()
    pp_df["n_labels"] = pp_df["label_list"].apply(len)

    # Keep only single-label images for clean classification
    single_label = pp_df[pp_df["n_labels"] == 1].copy()
    single_label["label"] = single_label["label_list"].apply(lambda x: x[0])
    single_label["our_class"] = single_label["label"].map(PP2021_CLASS_MAP)

    print(f"Total images:        {len(pp_df)}")
    print(f"Single-label images: {len(single_label)}")
    print(f"Multi-label (skipped): {len(pp_df) - len(single_label)}")
    print()
    print("Class distribution (single-label only):")
    print(single_label.groupby(["label", "our_class"]).size().to_string())
except Exception as e:
    print(f"Could not parse Plant Pathology CSV: {e}")
    print("Check that PLANT_PATH is correct and the CSV has a 'labels' column.")

In [ ]:
# Plot: Plant Pathology 2021 class distribution
try:
    class_counts = single_label["our_class"].value_counts()

    ax = class_counts.plot(kind="bar", color="#2A9D8F", edgecolor="white")
    plt.title("Plant Pathology 2021 — Class Distribution (single-label)", fontsize=14, fontweight='bold')
    plt.xlabel("Class")
    plt.ylabel("Image Count")
    plt.xticks(rotation=30, ha='right')
    for p in ax.patches:
        ax.annotate(str(int(p.get_height())),
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom', fontsize=10)
    plt.tight_layout()
    plt.show()
except Exception:
    print("Run the cell above first (needs Plant Pathology CSV loaded).")

In [ ]:
# Sample images from Plant Pathology 2021
try:
    img_dir = PLANT_PATH / "train_images"
    classes_present = single_label["our_class"].dropna().unique()
    n_cols = 4
    n_rows = len(classes_present)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
    fig.suptitle("Plant Pathology 2021 — Sample Images per Class", fontsize=14, fontweight='bold')

    for row_idx, cls in enumerate(sorted(classes_present)):
        subset = single_label[single_label["our_class"] == cls]
        samples = subset.sample(min(n_cols, len(subset)))
        for col_idx, (_, row) in enumerate(samples.iterrows()):
            ax = axes[row_idx][col_idx] if n_rows > 1 else axes[col_idx]
            img_path = img_dir / row["image"]  # adjust column name if needed
            if img_path.exists():
                img = Image.open(img_path)
                ax.imshow(img)
                ax.set_title(f"{cls}\n{img.size[0]}x{img.size[1]}", fontsize=8)
            else:
                ax.set_title(f"{cls}\n(not found)", fontsize=8)
            ax.axis('off')

    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Could not show Plant Pathology samples: {e}")

---
## 5. Combined Dataset Overview
Summary of what both datasets contribute to our final training set.

In [ ]:
summary = []

# NVIDIA contribution
for _, row in apple_nvidia[apple_nvidia["split"] == "train"].iterrows():
    summary.append({"class": row["our_class"], "source": "NVIDIA Fruits", "count": row["count"]})

# Plant Pathology contribution
try:
    for cls, cnt in single_label["our_class"].value_counts().items():
        summary.append({"class": cls, "source": "Plant Pathology 2021", "count": cnt})
except Exception:
    pass

summary_df = pd.DataFrame(summary)

if not summary_df.empty:
    pivot = summary_df.pivot_table(index="class", columns="source", values="count", fill_value=0)
    pivot["TOTAL"] = pivot.sum(axis=1)
    print(pivot.sort_values("TOTAL", ascending=False).to_string())
    print(f"\nGrand total images: {pivot['TOTAL'].sum()}")

    # Stacked bar
    pivot.drop(columns="TOTAL").plot(kind="bar", stacked=True,
                                      color=["#4C9BE8", "#2A9D8F"], edgecolor="white")
    plt.title("Combined Dataset — Images per Class", fontsize=14, fontweight='bold')
    plt.xlabel("Class")
    plt.ylabel("Image Count")
    plt.xticks(rotation=30, ha='right')
    plt.legend(title="Source")
    plt.tight_layout()
    plt.show()
else:
    print("No combined data yet — make sure both dataset paths are correct.")

---
## 6. Data Quality Checks

In [ ]:
# Check for corrupt/unreadable images in NVIDIA apple folders
corrupted = []
checked = 0

for orig_cls in NVIDIA_CLASS_MAP:
    for split in ["train", "valid"]:
        cls_path = NVIDIA_FRUITS_PATH / split / orig_cls
        if not cls_path.exists():
            continue
        for img_path in cls_path.glob("*.png"):
            checked += 1
            try:
                Image.open(img_path).verify()
            except Exception as e:
                corrupted.append({"path": str(img_path), "error": str(e)})

print(f"Checked {checked} NVIDIA apple images")
print(f"Corrupted: {len(corrupted)}")
if corrupted:
    for c in corrupted:
        print(f"  {c['path']} — {c['error']}")

In [ ]:
# Class imbalance check — flag classes with fewer than 100 images
MINIMUM_IMAGES = 100

print("Class imbalance report:")
if not summary_df.empty:
    totals = summary_df.groupby("class")["count"].sum()
    for cls in ALL_CLASSES:
        count = totals.get(cls, 0)
        flag = " ⚠️  UNDER-REPRESENTED" if count < MINIMUM_IMAGES else ""
        print(f"  {cls:<30} {count:>5} images{flag}")
else:
    print("Run section 5 first.")

---
## 7. Next Steps

Once this notebook is complete:
1. Run **`python src/preprocess.py`** — merges both datasets into `data/processed/` in YOLOv8-cls format
2. Run **`python src/train.py --task classify --data data/processed --epochs 10`** — first training run
3. Weights will be saved to `models/` and the server can load them via `predict.py`

**Notes for Yacine:**
- `bruised_fruit` has no source dataset yet — needs manual collection or augmentation
- `apple_black_rot` may be scarce in Plant Pathology 2021 — flag this after running section 6
- If class imbalance is severe, `preprocess.py` will apply augmentation to the minority classes